## Data Ingestion Process

In [1]:
import os
# os.chdir('../')
%pwd

'c:\\Users\\singh\\OneDrive\\Desktop\\TextSummarizer'

## Data Configuartion
`src/textsummarizer/entity/config_entity.py`

In [2]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

`src/textsummarizer/config/configuration.py`

In [3]:
from src.textsummarizer.constants import *
from src.textsummarizer.utils.main import read_yaml, create_directories

In [4]:
class ConfigurationManager:
    def __init__(self, 
                 config_path=CONFIG_FILE_PATH,
                 params_path=PARAMS_FILE_PATH):
        self.config = read_yaml(config_path)
        self.params = read_yaml(params_path)

        create_directories([self.config.artifacts_root])
    
    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion
        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir
        )
        return data_ingestion_config

## Components
`src/textsummarizer/components/dataingestion.py`

In [5]:
import urllib.request as request
import zipfile
from src.textsummarizer.logging import logger

class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    ## Downloading the Zip file
    def download_zip(self):
        if not os.path.exists(self.config.local_data_file):
            filename, header = request.urlretrieve(
                url= self.config.source_URL,
                filename= self.config.local_data_file
            )
            logger.info(f"{filename} Downloaded! with following info: \n{header}")
        else:
            logger.info(f"File already exists")
    
    ## Extracting the Zip file
    def extract_zip(self):
        """
        zip_file_path: str
        Extracts the zip file into the data directory
        Function returns None
        """

        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_file:
            zip_file.extractall(unzip_path)

In [7]:
try:
    config=ConfigurationManager()
    data_ingestion_config=config.get_data_ingestion_config()
    data_ingestion=DataIngestion(config=data_ingestion_config)
    data_ingestion.download_zip()
    data_ingestion.extract_zip()
except Exception as e:
    raise e

[ 2026-05-13 20:23:51,568 : INFO : main : YAML file: config\config.yaml Loaded Successfully ]
[ 2026-05-13 20:23:51,573 : INFO : main : YAML file: params.yaml Loaded Successfully ]
[ 2026-05-13 20:23:51,578 : INFO : main : Create Directory at: artifacts ]
[ 2026-05-13 20:23:51,579 : INFO : main : Create Directory at: artifacts/data_ingestion ]
[ 2026-05-13 20:23:54,841 : INFO : 931768761 : artifacts/data_ingestion/data.zip Downloaded! with following info: 
Connection: close
Content-Length: 7903594
Cache-Control: max-age=300
Content-Security-Policy: default-src 'none'; style-src 'unsafe-inline'; sandbox
Content-Type: application/zip
ETag: "dbc016a060da18070593b83afff580c9b300f0b6ea4147a7988433e04df246ca"
Strict-Transport-Security: max-age=31536000
X-Content-Type-Options: nosniff
X-Frame-Options: deny
X-XSS-Protection: 1; mode=block
X-GitHub-Request-Id: 2006:3363CD:5F9D93:6D279D:6A04907F
Accept-Ranges: bytes
Date: Wed, 13 May 2026 14:53:53 GMT
Via: 1.1 varnish
X-Served-By: cache-del-vibw